# Mini Example — XGBoost Solar Forecasting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akekapong78/ai-competition/blob/main/01-forecasting/mini_example.ipynb)

รันครบใน ~2 นาที | ไม่มี external data | ทดสอบ pipeline ก่อนใช้ data จริง

In [ ]:
!pip install xgboost -q
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
print('Ready')

In [ ]:
# --- 1. Data ---
idx = pd.date_range('2023-01-01', '2024-12-31', freq='1h')
np.random.seed(42)
h = idx.hour
solar = np.maximum(0, 10 * np.sin(np.pi*(h-6)/12) * (1+0.3*np.cos(2*np.pi*idx.month/12)) + np.random.normal(0,0.3,len(idx)))
df = pd.DataFrame({'solar_mw': solar}, index=idx)
df.head()

In [ ]:
# --- 2. Features ---
def make_features(df):
    d = df.copy()
    t = d['solar_mw']
    d['hour']       = d.index.hour
    d['month']      = d.index.month
    d['dow']        = d.index.dayofweek
    d['hour_sin']   = np.sin(2*np.pi*d['hour']/24)
    d['hour_cos']   = np.cos(2*np.pi*d['hour']/24)
    d['is_day']     = ((d['hour']>=6)&(d['hour']<=18)).astype(int)
    d['solar_elev'] = np.maximum(0, np.sin(np.pi*(d['hour']-6)/12))
    for lag in [1,2,6,12,24,48,168]:
        d[f'lag_{lag}'] = t.shift(lag)
    d['roll_mean_24'] = t.shift(1).rolling(24).mean()
    d['roll_std_24']  = t.shift(1).rolling(24).std()
    return d.dropna()

data = make_features(df)
FEATS  = [c for c in data.columns if c != 'solar_mw']
print(f'Features: {len(FEATS)} | Rows: {len(data):,}')

In [ ]:
# --- 3. Train/Test split (chronological) ---
train = data[data.index < '2024-06-01']
test  = data[data.index >= '2024-06-01']
print(f'Train: {len(train):,} | Test: {len(test):,}')

In [ ]:
# --- 4. Train ---
model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
model.fit(train[FEATS], train['solar_mw'],
          eval_set=[(test[FEATS], test['solar_mw'])], verbose=False)
print('Done')

In [ ]:
# --- 5. Evaluate ---
pred = np.maximum(0, model.predict(test[FEATS]))
mae  = mean_absolute_error(test['solar_mw'], pred)
rmse = np.sqrt(mean_squared_error(test['solar_mw'], pred))
mask = test['solar_mw'] > 0.1
mape = np.mean(np.abs((test['solar_mw'][mask]-pred[mask])/test['solar_mw'][mask]))*100
print(f'MAE={mae:.3f} MW | RMSE={rmse:.3f} MW | MAPE={mape:.1f}%')

In [ ]:
# --- 6. Plot 5 วัน ---
n = 5*24
fig, ax = plt.subplots(figsize=(14,4))
ax.plot(test.index[:n], test['solar_mw'].values[:n], label='Actual',    color='orange', lw=2)
ax.plot(test.index[:n], pred[:n],                    label='Predicted', color='blue',   lw=1.5, ls='--')
ax.set_title('Solar Forecast vs Actual')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- 7. Submission function (ใช้วันแข่ง) ---
def fill_missing(csv_in, csv_out, model, feats_fn=make_features):
    df_raw = pd.read_csv(csv_in, parse_dates=['datetime'], index_col='datetime')
    feats  = feats_fn(df_raw[['solar_mw']])
    fcols  = [c for c in feats.columns if c != 'solar_mw']
    missing = df_raw['solar_mw'].isna()
    pred = np.maximum(0, model.predict(feats.loc[missing, fcols]))
    df_raw.loc[missing, 'solar_mw'] = pred
    df_raw.to_csv(csv_out)
    print(f'Filled {missing.sum()} rows → {csv_out}')

# fill_missing('competition_data.csv', 'submission.csv', model)
print('Pipeline ready')